In [2]:
import requests
from bs4 import BeautifulSoup
import csv
import re 
import pandas as pd
import numpy as np
import math
# Suppress just SettingWithCopyWarning
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.ChainedAssignmentError)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
pd.options.mode.chained_assignment = None  # Disable the warning
import time
import json


## Part 1: 

Final data repairs before moving to clustering

### Part 1: Fixing current_source_airport_details.csv 

There are still errors here, as some columns have a coordinate that is 0,0, which is clearly invalid

In [4]:
data= pd.read_csv("./data/current_source_airports_details.csv")
print("original data length:", len(data))

original data length: 980


In [5]:
for index, row in data.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    if(lat == lon == 0):
        print("invalid coordinates for Iata:", row["IATA"])

These rows were manually fixed

### Part 2: getting updated current routes data

Using the same code as the old part 2:

In [4]:
def get_destinations(iata_source, wiki_name, path_write):
    file = open(path_write, "a") #file to append to
    
    url = f"https://en.wikipedia.org/wiki/{wiki_name}"
    response = requests.get(url)
    
    soup = BeautifulSoup(response.text, 'html.parser')
    #find the related destination table
    # Case-insensitive string match
    heading = soup.find("h2", string=re.compile(r"destination", re.IGNORECASE))
    #check text in heading
    heading_text =  heading.get_text()
    if  "former" in heading_text or "Former" in heading_text: #if either text is found, abort the function. This indicate the airport is no longer in service
        file.close() #close
        return
        
    
    table = heading.find_next("table") 
    while ('wikitable' not in table.get("class")): #find the next table matching a predictable class, if one has not been found
        table = table.find_next("table") 
    rows = table.find_all("tr")

    
    for i in range(1,len(rows)): #exclude the first row
        row = rows[i]
        # Extract all cells (td or th)
        cols = row.find_all(["td", "th"])
        # Write the row text content to CSV
        #first column is the airline
        airline = cols[0].get_text(strip=True)
        #get the list of destinations in the 2nd  
        destinations = cols[1]
        isSeasonal = 0 #iterate over subcomponents (seasonal always comes last, so set is seasonal to be false for now)
        for child in destinations.children: 
            #anchor components are the only destinations
            if (child.name == "a"):
                dest_name = child.get('title') #the title is the official wikipedia airport name (without _ in place of spaces)
                dest_name = dest_name.replace(" ", "_") 
                output = f"\"{iata_source}\",\"{wiki_name}\",\"{dest_name}\",\"{airline}\",\"{isSeasonal}\"\n" #final output to append to the file
                file.write(output)#write file
            elif ((child.name == "b") and (child.text == "Seasonal:")):
                isSeasonal = 1 #get seasonal to be 1 for future destinations
    file.close() #close
    return

Now, since we have a better source airports details data, we can try to get better current routes data using redirects. This allows clustering and accurate Dijkstra's algorithm to be ran

In [5]:
f = open("./data/current_routes.csv", "w")
f.write("iata_source,starting_wiki_name,dest_wikipedia_name,airline,isSeasonal\n") 
f.close() #add column names

In [6]:
codes_list = data["IATA"]
names_list = data["wiki_name"]
alt_names_list = data["redirects"] #alternative name
file_append_path = "./data/current_routes.csv"

Testing when alternative redirect would be used

In [7]:
#iterate based on details 
for i in range(0,len(data)):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        print("alt name used:", alt, "for original name:",name)
        name=alt #use alternative name if the alt is empty
        


airport index: 0
airport index: 1
airport index: 2
airport index: 3
airport index: 4
airport index: 5
airport index: 6
airport index: 7
airport index: 8
airport index: 9
airport index: 10
airport index: 11
airport index: 12
airport index: 13
airport index: 14
airport index: 15
airport index: 16
airport index: 17
airport index: 18
airport index: 19
airport index: 20
airport index: 21
airport index: 22
airport index: 23
alt name used: Madrid–Barajas_Airport for original name: Adolfo_Suárez_Madrid–Barajas_Airport
airport index: 24
airport index: 25
airport index: 26
airport index: 27
airport index: 28
airport index: 29
airport index: 30
airport index: 31
airport index: 32
airport index: 33
airport index: 34
airport index: 35
airport index: 36
airport index: 37
airport index: 38
airport index: 39
airport index: 40
airport index: 41
airport index: 42
airport index: 43
airport index: 44
airport index: 45
airport index: 46
airport index: 47
airport index: 48
airport index: 49
airport index: 5

iterrate through airports 0 to 50

In [8]:
#iterate based on details 
for i in range(0,50):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")


airport index: 0
airport index: 1
airport index: 2
airport index: 3
airport index: 4
airport index: 5
airport index: 6
airport index: 7
airport index: 8
airport index: 9
airport index: 10
airport index: 11
airport index: 12
airport index: 13
airport index: 14
airport index: 15
airport index: 16
airport index: 17
airport index: 18
airport index: 19
airport index: 20
airport index: 21
airport index: 22
airport index: 23
airport index: 24
airport index: 25
airport index: 26
airport index: 27
airport index: 28
airport index: 29
airport index: 30
airport index: 31
airport index: 32
airport index: 33
airport index: 34
airport index: 35
airport index: 36
airport index: 37
airport index: 38
airport index: 39
airport index: 40
airport index: 41
airport index: 42
airport index: 43
airport index: 44
airport index: 45
airport index: 46
airport index: 47
airport index: 48
airport index: 49


continue iterating for each step 

In [9]:
#iterate based on details 
for i in range(50,150):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")


airport index: 50
airport index: 51
airport index: 52
airport index: 53
airport index: 54
airport index: 55
airport index: 56
airport index: 57
airport index: 58
airport index: 59
airport index: 60
airport index: 61
airport index: 62
airport index: 63
airport index: 64
airport index: 65
airport index: 66
airport index: 67
airport index: 68
airport index: 69
airport index: 70
airport index: 71
airport index: 72
airport index: 73
airport index: 74
airport index: 75
airport index: 76
airport index: 77
airport index: 78
airport index: 79
airport index: 80
airport index: 81
airport index: 82
airport index: 83
airport index: 84
airport index: 85
airport index: 86
airport index: 87
airport index: 88
airport index: 89
airport index: 90
airport index: 91
airport index: 92
airport index: 93
airport index: 94
airport index: 95
airport index: 96
airport index: 97
airport index: 98
airport index: 99
airport index: 100
airport index: 101
airport index: 102
airport index: 103
airport index: 104
airpo

In [10]:
#iterate based on details 
for i in range(150,250):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")


airport index: 150
airport index: 151
airport index: 152
airport index: 153
airport index: 154
airport index: 155
airport index: 156
airport index: 157
airport index: 158
airport index: 159
airport index: 160
airport index: 161
airport index: 162
airport index: 163
airport index: 164
airport index: 165
airport index: 166
airport index: 167
airport index: 168
airport index: 169
airport index: 170
airport index: 171
airport index: 172
airport index: 173
airport index: 174
airport index: 175
airport index: 176
airport index: 177
airport index: 178
airport index: 179
airport index: 180
airport index: 181
airport index: 182
airport index: 183
airport index: 184
airport index: 185
airport index: 186
airport index: 187
airport index: 188
airport index: 189
airport index: 190
airport index: 191
airport index: 192
airport index: 193
airport index: 194
airport index: 195
airport index: 196
airport index: 197
airport index: 198
airport index: 199
airport index: 200
airport index: 201
airport inde

In [11]:
#iterate based on details 
for i in range(250,350):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")


airport index: 250
airport index: 251
airport index: 252
airport index: 253
airport index: 254
airport index: 255
airport index: 256
airport index: 257
airport index: 258
airport index: 259
airport index: 260
airport index: 261
airport index: 262
airport index: 263
airport index: 264
airport index: 265
airport index: 266
airport index: 267
airport index: 268
airport index: 269
airport index: 270
airport index: 271
airport index: 272
airport index: 273
airport index: 274
airport index: 275
airport index: 276
airport index: 277
airport index: 278
airport index: 279
airport index: 280
airport index: 281
airport index: 282
airport index: 283
airport index: 284
airport index: 285
airport index: 286
airport index: 287
airport index: 288
airport index: 289
airport index: 290
airport index: 291
airport index: 292
airport index: 293
airport index: 294
airport index: 295
airport index: 296
airport index: 297
airport index: 298
airport index: 299
airport index: 300
airport index: 301
airport inde

In [12]:
#iterate based on details 
for i in range(350,450):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")


airport index: 350
airport index: 351
airport index: 352
airport index: 353
airport index: 354
airport index: 355
airport index: 356
airport index: 357
airport index: 358
airport index: 359
airport index: 360
airport index: 361
airport index: 362
airport index: 363
airport index: 364
airport index: 365
airport index: 366
airport index: 367
airport index: 368
airport index: 369
airport index: 370
airport index: 371
airport index: 372
airport index: 373
airport index: 374
airport index: 375
airport index: 376
airport index: 377
airport index: 378
airport index: 379
airport index: 380
airport index: 381
airport index: 382
airport index: 383
airport index: 384
airport index: 385
airport index: 386
airport index: 387
airport index: 388
airport index: 389
airport index: 390
airport index: 391
airport index: 392
airport index: 393
airport index: 394
airport index: 395
airport index: 396
airport index: 397
airport index: 398
airport index: 399
airport index: 400
airport index: 401
airport inde

In [13]:
#iterate based on details 
for i in range(450,550):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

airport index: 450
airport index: 451
airport index: 452
airport index: 453
airport index: 454
airport index: 455
airport index: 456
airport index: 457
airport index: 458
airport index: 459
airport index: 460
airport index: 461
airport index: 462
airport index: 463
airport index: 464
airport index: 465
airport index: 466
airport index: 467
airport index: 468
airport index: 469
airport index: 470
airport index: 471
airport index: 472
airport index: 473
airport index: 474
airport index: 475
airport index: 476
airport index: 477
airport index: 478
airport index: 479
airport index: 480
airport index: 481
airport index: 482
airport index: 483
airport index: 484
airport index: 485
airport index: 486
airport index: 487
airport index: 488
airport index: 489
airport index: 490
airport index: 491
airport index: 492
airport index: 493
airport index: 494
airport index: 495
airport index: 496
airport index: 497
airport index: 498
airport index: 499
airport index: 500
airport index: 501
airport inde

In [14]:
#iterate based on details 
for i in range(550,650):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

airport index: 550
airport index: 551
airport index: 552
airport index: 553
airport index: 554
airport index: 555
airport index: 556
airport index: 557
airport index: 558
airport index: 559
airport index: 560
airport index: 561
airport index: 562
airport index: 563
airport index: 564
airport index: 565
airport index: 566
airport index: 567
airport index: 568
airport index: 569
airport index: 570
airport index: 571
airport index: 572
airport index: 573
airport index: 574
airport index: 575
airport index: 576
airport index: 577
airport index: 578
airport index: 579
airport index: 580
airport index: 581
airport index: 582
airport index: 583
airport index: 584
airport index: 585
airport index: 586
airport index: 587
airport index: 588
airport index: 589
airport index: 590
airport index: 591
airport index: 592
airport index: 593
airport index: 594
airport index: 595
airport index: 596
airport index: 597
airport index: 598
airport index: 599
airport index: 600
airport index: 601
airport inde

In [15]:
#iterate based on details 
for i in range(650,750):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

airport index: 650
airport index: 651
airport index: 652
airport index: 653
airport index: 654
airport index: 655
airport index: 656
airport index: 657
airport index: 658
airport index: 659
airport index: 660
airport index: 661
airport index: 662
airport index: 663
airport index: 664
airport index: 665
airport index: 666
airport index: 667
airport index: 668
airport index: 669
airport index: 670
airport index: 671
airport index: 672
airport index: 673
airport index: 674
airport index: 675
airport index: 676
airport index: 677
airport index: 678
airport index: 679
airport index: 680
airport index: 681
airport index: 682
airport index: 683
airport index: 684
airport index: 685
airport index: 686
airport index: 687
airport index: 688
airport index: 689
airport index: 690
airport index: 691
airport index: 692
airport index: 693
airport index: 694
airport index: 695
airport index: 696
airport index: 697
airport index: 698
airport index: 699
airport index: 700
airport index: 701
airport inde

In [16]:
#iterate based on details 
for i in range(750,850):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

airport index: 750
airport index: 751
airport index: 752
airport index: 753
airport index: 754
airport index: 755
airport index: 756
airport index: 757
airport index: 758
airport index: 759
airport index: 760
airport index: 761
airport index: 762
airport index: 763
airport index: 764
airport index: 765
airport index: 766
airport index: 767
airport index: 768
airport index: 769
airport index: 770
airport index: 771
airport index: 772
airport index: 773
airport index: 774
airport index: 775
airport index: 776
airport index: 777
airport index: 778
airport index: 779
airport index: 780
airport index: 781
airport index: 782
airport index: 783
airport index: 784
airport index: 785
airport index: 786
airport index: 787
airport index: 788
airport index: 789
airport index: 790
airport index: 791
airport index: 792
airport index: 793
airport index: 794
airport index: 795
airport index: 796
airport index: 797
airport index: 798
airport index: 799
airport index: 800
airport index: 801
airport inde

In [17]:
#iterate based on details 
for i in range(850,950):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

airport index: 850
airport index: 851
airport index: 852
airport index: 853
airport index: 854
airport index: 855
airport index: 856
airport index: 857
airport index: 858
airport index: 859
airport index: 860
airport index: 861
airport index: 862
airport index: 863
airport index: 864
airport index: 865
airport index: 866
airport index: 867
airport index: 868
airport index: 869
airport index: 870
airport index: 871
airport index: 872
airport index: 873
airport index: 874
airport index: 875
airport index: 876
airport index: 877
airport index: 878
airport index: 879
airport index: 880
airport index: 881
airport index: 882
airport index: 883
airport index: 884
airport index: 885
airport index: 886
airport index: 887
airport index: 888
airport index: 889
airport index: 890
airport index: 891
airport index: 892
airport index: 893
airport index: 894
airport index: 895
airport index: 896
airport index: 897
airport index: 898
airport index: 899
airport index: 900
airport index: 901
airport inde

In [18]:
#iterate based on details 
for i in range(950,len(data)):
    print("airport index:", i)
    code = codes_list[i]
    name = names_list[i]
    alt = alt_names_list[i]
    if(alt and str(alt)!="nan" and str(alt)!=""):
        name=alt #use alternative name if the alt is empty
    try:
        get_destinations(code,name,file_append_path)
    except:
        print(f"failed airport:{code}")

airport index: 950
airport index: 951
airport index: 952
airport index: 953
airport index: 954
airport index: 955
airport index: 956
airport index: 957
airport index: 958
airport index: 959
airport index: 960
airport index: 961
airport index: 962
airport index: 963
airport index: 964
airport index: 965
airport index: 966
airport index: 967
airport index: 968
airport index: 969
airport index: 970
airport index: 971
airport index: 972
airport index: 973
airport index: 974
airport index: 975
airport index: 976
airport index: 977
airport index: 978
airport index: 979


Checking for missing airports 


In [10]:
routes = pd.read_csv("./data/current_routes.csv")
unique_airports_in_routes = set(routes["iata_source"].unique())
#find missing airports
missing = set(codes_list) - unique_airports_in_routes
print("")
for m in missing:
    print("missing airports", m)

Fixing faulty airline names 


In [11]:
routes = pd.read_csv("./data/current_routes.csv", encoding="utf-8")

for index, row in routes.iterrows():
    airline = row["airline"]
    list = airline.split("[")
    if (len(list)>1):
        print("fixing airlines name:", airline)
        routes.at[index, "airline"] = list[0]


fixing airlines name: Enter Air[191][192]
fixing airlines name: Aer Lingus[93]
fixing airlines name: Aer Lingus[93]
fixing airlines name: Aer Lingus[93]
fixing airlines name: Aer Lingus[93]
fixing airlines name: GP Aviation[175]
fixing airlines name: Tus Airways[233]
fixing airlines name: Aegean Airlines[68]
fixing airlines name: Aegean Airlines[68]
fixing airlines name: Aer Lingus[71]
fixing airlines name: Aeroméxico[73]
fixing airlines name: Air Arabia[76]
fixing airlines name: Air Arabia[76]
fixing airlines name: Air Arabia[76]
fixing airlines name: Air Arabia[76]
fixing airlines name: Air Astana[81]
fixing airlines name: Air Canada[82]
fixing airlines name: Air Canada[82]
fixing airlines name: Air Europa[86]
fixing airlines name: Air France[88]
fixing airlines name: Air France[88]
fixing airlines name: Air Serbia[92]
fixing airlines name: Air Transat[94]
fixing airlines name: airBaltic[96]
fixing airlines name: airBaltic[96]
fixing airlines name: airBaltic[96]
fixing airlines name:

In [12]:
routes.to_csv("./data/current_routes.csv", encoding="utf-8")

## Part 2: Isolate routes between the top 980 airports (all airports in source data)



In [13]:
routes = pd.read_csv("./data/current_routes.csv", encoding='utf-8')
routes.head(n=1)

,Unnamed: 0,iata_source,starting_wiki_name,dest_wikipedia_name,airline,isSeasonal
0,0,ATL,Hartsfield–Jackson_Atlanta_International_Airport,Bajío_International_Airport,Aeroméxico Connect,0


In [10]:
airport_data = pd.read_csv("./data/current_source_airports_details.csv", encoding='utf-8')
airport_data.head(n=1)

,IATA,wiki_name,city,country,latitude,longitude,redirects,pre2020_ids,pre2022_ids
0,ATL,Hartsfield–Jackson_Atlanta_International_Airport,Atlanta,Usa,33.64,-84.427,NaN,932935279,1063034925


In [18]:
#isolate directs wikinames sets 
wikinames =set(airport_data["wiki_name"])
redirects =set(airport_data["redirects"])
unique_iatas = set(airport_data["IATA"])

get list of rows by indices to keep

In [15]:
keep_indices = []
dest_iatas = [] #get list of corresponding dest iata codes
for index, row in routes.iterrows():
    currdest = row["dest_wikipedia_name"]
    if (currdest in wikinames): 
        #get matching iata codes
        match = airport_data[airport_data["wiki_name"]==currdest].iloc[0]
        match = match["IATA"]
        keep_indices.append(index)
        dest_iatas.append(match)
    elif (currdest in redirects): 
        #get matching iata codes
        match = airport_data[airport_data["redirects"]==currdest].iloc[0]
        match = match["IATA"]
        keep_indices.append(index)
        dest_iatas.append(match)
    else: 
        continue
print("length of destination iatas", len(dest_iatas))
print("number of unique  destination iatas", len(set(dest_iatas)))

length of destination iatas 52433
number of unique  destination iatas 966


create unique route data

In [16]:
new_data = routes.iloc[keep_indices]
new_data["dest_IATA"] = dest_iatas

In [17]:
new_data.head(n=2)

,Unnamed: 0,iata_source,starting_wiki_name,dest_wikipedia_name,airline,isSeasonal,dest_IATA
2,2,ATL,Hartsfield–Jackson_Atlanta_International_Airport,Monterrey_International_Airport,Aeroméxico Connect,0,MTY
5,5,ATL,Hartsfield–Jackson_Atlanta_International_Airport,Toronto_Pearson_International_Airport,Air Canada,0,YYZ


print missing iatas:

In [19]:
missing_iata = unique_iatas-set(dest_iatas)
print("missing iatas", missing_iata)

missing iatas {'LGP', 'AAQ', 'KBP', 'FXE', 'MUN', 'SIC', 'AGT', 'ADA', 'DKR', 'BSL', 'PAC', 'WJR', 'KRT', 'GDZ'}


Save the new data since these are all small airports

In [20]:
new_data.to_json('key_airports_network.json', orient='records', indent=4)

# Part 3:
### Data exports. Now, export our special routes data, the routes changes by airline and airport datasets (exported from powerbi) to json

In [4]:
data = pd.read_csv("./data/airport_changes.csv", encoding="utf-8")
print(len(data))
data.head(n=1)

978


,iata_source,Route_count,pre_2022_route_count,pre_2020_route_count,current_source_airports_details.city,current_source_airports_details.country,current_source_airports_details.latitude,current_source_airports_details.longitude,current_vs_pre2020_routes,pre2022_vs_pre2020_routes,current_vs_pre2022_routes
0,FUK,110,110,110,Fukuoka,Japan,33.586,130.45,0,0,0


In [6]:
data.to_json('airport_changes.json', orient='records', indent=4)

In [7]:
data = pd.read_csv("./data/airline_changes.csv", encoding="utf-8")
print(len(data))
data.head(n=1)
data.to_json('airline_changes.json', orient='records', indent=4)

590
